# Proyecto CallMeMaybe — Descomposición de tareas 

**Objetivo:** definir el plan de trabajo para identificar operadores ineficaces y comunicar conclusiones accionables.

**Entregables del caso principal:**
- **Descomposición de tareas**
- **Implementación del plan**

## 1. Comprensión del problema

### 1.1 Contexto
El servicio de telefonía virtual **CallMeMaybe** está desarrollando una función para ayudar a supervisores a detectar **operadores menos eficaces**.

### 1.2 Definición operativa de “operador ineficaz” 
Un operador se considera ineficaz si presenta una o más de las siguientes condiciones:

- **Alta cantidad/tasa de llamadas entrantes perdidas**, tanto **internas** como **externas**.
- **Tiempo de espera prolongado** en llamadas entrantes.
- Si el operador **debe realizar llamadas salientes**, entonces un **número reducido** de llamadas salientes también es señal de ineficacia.

### 1.3 Resultado esperado
Construir un método reproducible para:
1) Analizar la operación (EDA),
2) Definir métricas y criterios para etiquetar operadores ineficaces,
3) Validar hallazgos con pruebas estadísticas,
4) Comunicar conclusiones y recomendaciones en una presentación ejecutiva (PDF).


## 2. Comprensión y evaluación de los datos

### 2.1 Fuentes de datos
Se utilizarán dos datasets:

**A) `telecom_dataset_us.csv`** (registro de llamadas)
- `user_id`: ID de la cuenta del cliente
- `date`: fecha de las estadísticas
- `direction`: dirección de la llamada (`in` entrante, `out` saliente)
- `internal`: si la llamada fue interna (entre operadores)
- `operator_id`: identificador del operador
- `is_missed_call`: indicador de llamada perdida
- `calls_count`: número de llamadas
- `call_duration`: duración de la llamada (sin tiempo de espera)
- `total_call_duration`: duración total (incluye tiempo de espera)

**B) `telecom_clients_us.csv`** (clientes)
- `user_id`: ID de usuario/a
- `tariff_plan`: tarifa actual
- `date_start`: fecha de registro

### 2.2 Validaciones iniciales planeadas
- Verificar tipos de datos y formato de fechas.
- Revisar valores faltantes y rangos fuera de lo esperado (duraciones negativas, `calls_count` en cero, etc.).
- Confirmar llaves de unión: `user_id` en ambos datasets y `operator_id` en el dataset de llamadas.
- Revisar duplicados por combinación (`user_id`, `operator_id`, `date`, `direction`, `internal`) si aplica.

### 2.3 Supuestos y limitaciones 
- Las métricas se calcularán **por operador** y **dentro de cada cliente (`user_id`)** para evitar comparaciones injustas entre empresas con volúmenes y procesos distintos.
- `calls_count` representa el volumen de llamadas agregado por fila; por lo tanto, se usarán **promedios ponderados** cuando corresponda.
- La variable “debe hacer llamadas salientes” no existe explícitamente; se inferirá con reglas observables (ver Sección 5).


## 3. Preparación de los datos

### 3.1 Estandarización de tipos
- Convertir `date` y `date_start` a formato datetime.
- Asegurar que `operator_id` sea numérico (y tratar nulos).
- Asegurar que `is_missed_call` y `internal` sean booleanos o 0/1 consistentes.

### 3.2 Limpieza
- Manejo de valores faltantes en columnas críticas (`operator_id`, `direction`, duraciones).
- Validación de reglas básicas:
  - `calls_count` > 0 (si no, justificar o filtrar)
  - `total_call_duration` ≥ `call_duration` (si no, investigar)
  - Duraciones no negativas

### 3.3 Métricas base
Se crearán métricas derivadas para evaluar desempeño:

- **Tiempo de espera total** por fila:
  - `wait_time_total = total_call_duration - call_duration`
- **Tiempo de espera promedio por llamada** (ponderado por `calls_count`):
  - `avg_wait = wait_time_total / calls_count`

### 3.4 Definición de granularidad
La unidad principal de evaluación será:

- **Operador dentro de cliente:** (`user_id`, `operator_id`)

Opcionalmente, se considerará agregar segmentación temporal (por semana/mes) si el comportamiento cambia sustancialmente con el tiempo.


## 4. Análisis exploratorio de datos (EDA)

### 4.1 Objetivos del EDA
- Entender el volumen y composición de llamadas.
- Identificar patrones temporales.
- Detectar outliers y posibles problemas de calidad de datos.
- Obtener una base para definir criterios de ineficacia (Sección 5).

### 4.2 Análisis descriptivo (global)
- Distribución de `direction` (entrante vs saliente).
- Distribución de `internal` (internas vs externas).
- Proporción global de `is_missed_call`.
- Distribuciones de `call_duration`, `total_call_duration`, `avg_wait`.

### 4.3 Análisis temporal
- Evolución diaria/semanal del volumen total de llamadas.
- Evolución de tasa de llamadas perdidas.
- Evolución del tiempo de espera promedio.

### 4.4 Análisis por cliente (`user_id`)
- Comparación de volúmenes y tasas por cliente.
- Identificar clientes con operación atípica (muy alta pérdida o alta espera).

### 4.5 Análisis por operador (`user_id`, `operator_id`)
- Ranking de operadores por:
  - tasa de llamadas perdidas entrantes
  - tiempo de espera promedio entrante
  - volumen de llamadas entrantes y salientes
- Evaluar si los “peores” operadores concentran el problema en pocos clientes.

### 4.6 Visualizaciones planeadas
- Histogramas / boxplots de `avg_wait` y `in_miss_rate`.
- Series temporales agregadas (calls, miss_rate, avg_wait).
- Heatmap o tabla resumen de clientes con mayor incidencia.


## 5. Definición de criterios de ineficacia

### 5.1 Métricas por operador (dentro de cada cliente)
Se calcularán, como mínimo:

**Para llamadas entrantes** (`direction == 'in'`):
- `in_calls`: total de llamadas entrantes
- `in_missed_calls`: total de llamadas entrantes perdidas
- `in_miss_rate = in_missed_calls / in_calls`
- `avg_wait_in`: tiempo de espera promedio por llamada entrante (ponderado)

**Para llamadas salientes** (`direction == 'out'`):
- `out_calls`: total de llamadas salientes

Además, se evaluará separación entre **internas** y **externas** en llamadas perdidas:
- `in_miss_rate_internal`
- `in_miss_rate_external`

### 5.2 Regla para identificar “operadores que deben hacer llamadas salientes”
Dado que el dataset no incluye un campo explícito de rol, se propondrán reglas observables, por ejemplo:

- Un operador se considera **outbound-capable** si registra al menos 1 llamada saliente en el periodo.
- Alternativa a evaluar: considerar outbound “requerido” cuando, a nivel cliente, una fracción relevante del tráfico es saliente (p. ej. ≥ 20% del total). En ese caso, se evalúan únicamente operadores con actividad saliente.

La regla final se elegirá tras el EDA para evitar sesgos.

### 5.3 Umbrales propuestos 
Los umbrales se definirán **relativos al cliente (`user_id`)** para comparaciones justas.

Propuesta inicial (sujeta a ajuste):
- Ineficaz por pérdidas si `in_miss_rate` está en el **cuartil superior** (≥ P75) dentro del cliente.
- Ineficaz por espera si `avg_wait_in` está en el **cuartil superior** (≥ P75) dentro del cliente.
- Ineficaz por salientes (si outbound-capable) si `out_calls` está en el **cuartil inferior** (≤ P25) dentro del cliente.

### 5.4 Control de ruido / mínimo de evidencia
Para reducir falsos positivos, se aplicará un umbral mínimo de actividad, por ejemplo:
- Considerar operadores con `in_calls >= N` (N definido tras revisar distribución; p. ej. 30).

### 5.5 Etiqueta final
Se creará una variable binaria:
- `ineffective_operator = 1` si cumple al menos una condición (pérdidas altas, espera alta, salientes bajos cuando aplica)
- `ineffective_operator = 0` en caso contrario


## 6. Identificación y análisis de operadores ineficaces

### 6.1 Aplicación del criterio
- Calcular métricas por (`user_id`, `operator_id`).
- Calcular percentiles/umbrales por `user_id`.
- Etiquetar operadores.

### 6.2 Resultados a reportar
- % de operadores ineficaces (global y por cliente).
- Distribución de causas de ineficacia:
  - principalmente por pérdidas
  - principalmente por espera
  - principalmente por salientes
  - combinaciones
- Top clientes con mayor concentración de operadores ineficaces.

### 6.3 Salidas operativas 
- Tabla priorizada de operadores con métricas clave y motivo(s) de ineficacia.
- Recomendación de seguimiento: coaching, revisión de enrutamiento, staffing por horarios, etc.


## 7. Pruebas de hipótesis estadísticas

### 7.1 Objetivo
Validar estadísticamente si los operadores clasificados como **ineficaces** presentan diferencias significativas frente a los **eficaces**, y si ciertos factores (p. ej., plan tarifario) se asocian con la ineficacia.

### 7.2 Hipótesis planificadas

#### Hipótesis 1 — Tiempo de espera (entrantes)
**Pregunta:** ¿Los operadores ineficaces generan mayor tiempo de espera en llamadas entrantes?

- **H₀:** La distribución (o mediana) de `avg_wait_in` es igual para operadores ineficaces y eficaces.
- **H₁:** `avg_wait_in` es mayor en operadores ineficaces.

**Test propuesto:** Mann–Whitney U (dos muestras independientes, no paramétrico).

---
#### Hipótesis 2 — Tasa de llamadas perdidas (entrantes)
**Pregunta:** ¿Los operadores ineficaces pierden más llamadas entrantes?

- **H₀:** La proporción/tasa de llamadas entrantes perdidas (`in_miss_rate`) es igual en operadores ineficaces y eficaces.
- **H₁:** `in_miss_rate` es mayor en operadores ineficaces.

**Test propuesto:** comparación de proporciones (Z-test) usando totales ponderados por `calls_count`.

---
#### Hipótesis 3 — Volumen de llamadas salientes (cuando aplica)
**Pregunta:** Cuando un operador se considera “outbound-capable”, ¿los ineficaces realizan menos llamadas salientes?

- **H₀:** La distribución (o mediana) de `out_calls` es igual en operadores ineficaces y eficaces (dentro del subconjunto outbound-capable).
- **H₁:** `out_calls` es menor en operadores ineficaces (dentro del subconjunto outbound-capable).

**Test propuesto:** Mann–Whitney U (no paramétrico) sobre el subconjunto outbound-capable.

---
#### Hipótesis 4 — Plan tarifario y probabilidad de ineficacia
**Pregunta:** ¿La tarifa (`tariff_plan`) se asocia con la presencia de operadores ineficaces?

- **H₀:** `tariff_plan` e `ineffective_operator` son independientes.
- **H₁:** Existe asociación entre `tariff_plan` e `ineffective_operator`.

**Test propuesto:** Chi-cuadrada de independencia (tabla de contingencia).

### 7.3 Consideraciones metodológicas
- Definir α (nivel de significancia), típicamente 0.05.
- Reportar: estadístico, p-value y conclusión (rechazo/no rechazo de H₀).
- Complementar significancia con tamaño de efecto cuando sea posible.


## 8. Interpretación, conclusiones y recomendaciones

### 8.1 Interpretación de resultados
- Traducir métricas y pruebas estadísticas a implicaciones operativas.
- Distinguir entre significancia estadística y relevancia práctica.

### 8.2 Conclusiones esperadas
- Principales drivers de ineficacia (pérdidas vs espera vs salientes).
- Clientes y periodos donde el problema se concentra.

### 8.3 Recomendaciones accionables
- Acciones por tipo de problema:
  - **Altas pérdidas:** revisión de cumplimiento de scripts, capacitación, revisión de enrutamiento, saturación por horario.
  - **Alta espera:** ajustes de staffing, redistribución de colas, reglas de enrutamiento.
  - **Bajas salientes (cuando aplica):** revisión de asignación de tareas, objetivos, horarios.
- Propuesta de KPI para monitoreo continuo en la nueva función.


## 9. Comunicación de resultados

### 9.1 Presentación ejecutiva (PDF)
Se preparará una presentación de 6–10 diapositivas que incluya:

1. Contexto y objetivo
2. Datos y preparación (resumen)
3. Métricas y criterio de ineficacia (con justificación)
4. Hallazgos clave (con números)
5. Resultados de hipótesis (p-values e interpretación)
6. Recomendaciones y próximos pasos

### 9.2 Lista de fuentes (5 a 10)
Se incluirá una sección de referencias con enlaces a documentación/artículos usados, y una breve nota indicando qué pregunta resolvió cada fuente (p. ej., elección de tests, definición de KPI, funciones de pandas utilizadas).

### 9.3 Enlace dentro del proyecto
El enlace al PDF final se colocará en el cuerpo del proyecto principal, según la instrucción del enunciado.
